# Spark SQL & Delta Lake Cheatsheet — Databricks Spark Associate Exam

**Exam Domain: Using Spark SQL (20%)**

## TEMP VIEWS

In [ ]:
# Session-scoped — disappears when session ends
df.createOrReplaceTempView("my_view")         # creates or replaces
df.createTempView("my_view")                  # errors if exists

# Application-scoped — survives across sessions in same app
df.createOrReplaceGlobalTempView("my_view")
# Access: SELECT * FROM global_temp.my_view   <-- prefix required!

# Drop views
spark.catalog.dropTempView("my_view")
spark.catalog.dropGlobalTempView("my_view")

## RUNNING SQL

In [ ]:
# spark.sql() returns a DataFrame — chain operations on it
result = spark.sql("SELECT * FROM my_view WHERE x > 5")
result.filter(result.y < 10).show()

## CATALOG OPERATIONS

In [ ]:
spark.catalog.listTables()                    # tables + temp views
spark.catalog.listColumns("table_name")       # column metadata
spark.catalog.tableExists("table_name")       # bool check

# SQL equivalents
# SHOW TABLES
# DESCRIBE TABLE table_name
# DESCRIBE EXTENDED table_name

## SQL QUERY PATTERNS

In [ ]:
# GROUP BY with HAVING
# SELECT col, COUNT(*) AS cnt FROM t GROUP BY col HAVING cnt > 10

# CTE (Common Table Expression)
# WITH cte AS (SELECT ... FROM t) SELECT * FROM cte WHERE ...

# Subquery
# SELECT * FROM t WHERE col > (SELECT AVG(col) FROM t)

# CASE expression
# SELECT CASE WHEN x > 10 THEN 'high' ELSE 'low' END AS label FROM t

# Window functions in SQL
# SELECT col, RANK() OVER (ORDER BY col DESC) AS rnk FROM t
# SELECT col, SUM(x) OVER (PARTITION BY g ORDER BY col) AS running FROM t

## DELTA LAKE

In [ ]:
# Time travel — read previous versions
# SELECT * FROM my_table VERSION AS OF 3
# SELECT * FROM my_table TIMESTAMP AS OF '2024-01-15'
spark.read.option("versionAsOf", 3).table("my_table")
spark.read.option("timestampAsOf", "2024-01-15").table("my_table")

# Restore to a previous version
# RESTORE TABLE my_table TO VERSION AS OF 3

# History — audit log of all changes
# DESCRIBE HISTORY my_table

## OPTIMIZE & Z-ORDER

In [ ]:
# Compact small files into larger ones
# OPTIMIZE my_table

# Co-locate data by column for faster filtering
# OPTIMIZE my_table ZORDER BY (col1, col2)

## VACUUM

In [ ]:
# Remove old files (default retention: 7 days / 168 hours)
# VACUUM my_table                          -- uses default retention
# VACUUM my_table RETAIN 168 HOURS        -- explicit retention
# VACUUM my_table DRY RUN                 -- preview without deleting
# WARNING: after VACUUM, time travel past retention is impossible

## MERGE INTO (Upsert)

In [ ]:
# MERGE INTO target USING source
# ON target.key = source.key
# WHEN MATCHED THEN UPDATE SET target.col = source.col
# WHEN NOT MATCHED THEN INSERT *
# WHEN NOT MATCHED BY SOURCE THEN DELETE

## SCHEMA COMMANDS

In [ ]:
# ALTER TABLE my_table ADD COLUMNS (new_col STRING)
# ALTER TABLE my_table RENAME COLUMN old_col TO new_col
# ALTER TABLE my_table DROP COLUMN col_name